In [ ]:
# ============================================================
# CELL 1 — Setup, Load & Data Validation
# Project: Bangla Cyberbullying Detection & Social Engagement Prediction
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, re, glob, os
pd.set_option('display.max_colwidth', 80)

PROJECT_DIR = '/content/drive/MyDrive/cyberbulling-detection'
RAW_DIR     = f'{PROJECT_DIR}/data/raw'

csv_files = glob.glob(f'{RAW_DIR}/*.csv')y
assert csv_files, f'No CSV found in {RAW_DIR}'
CSV_PATH = csv_files[0]
print(f'Loading: {os.path.basename(CSV_PATH)}\n')

df = pd.read_csv(CSV_PATH)

# ---------- 1. Structure ----------
print('='*60); print('1. STRUCTURE')
print(f'shape: {df.shape}')
print(df.dtypes.to_string())

# ---------- 2. Missing values ----------
print('\n'+'='*60); print('2. MISSING VALUES')
print(df.isna().sum().to_string())

# ---------- 3. Corrupted rows ----------
print('\n'+'='*60); print('3. CORRUPTED ROWS')
corrupt = df['Text'].astype(str).str.strip().isin(['#NAME?', '#REF!', '#VALUE!', ''])
print(f'Excel-error / empty Text: {corrupt.sum()}')

# ---------- 4. Duplicates ----------
print('\n'+'='*60); print('4. DUPLICATE ANALYSIS')
txt = df['Text'].astype(str).str.strip()
print(f'fully identical rows      : {df.duplicated().sum()}')
print(f'duplicated Text values    : {txt.duplicated().sum()}')

nlab = df.assign(_t=txt).groupby('_t')['Label'].nunique()
consistent_dup = df.assign(_t=txt)[df.assign(_t=txt)['_t'].isin(nlab[nlab == 1].index) & txt.duplicated(keep=False)]
conflicting    = nlab[nlab > 1].index
print(f'dup Text, SAME label      : {len(consistent_dup)} rows')
print(f'dup Text, CONFLICTING label: {len(conflicting)} unique texts '
      f'({df.assign(_t=txt)["_t"].isin(conflicting).sum()} rows)')
print('\n--- conflicting examples (strategy must be documented, do NOT drop silently) ---')
for t in list(conflicting)[:5]:
    print(f'  {t[:55]!r} -> {df.assign(_t=txt).query("_t == @t")["Label"].tolist()}')

# ---------- 5. Text length ----------
print('\n'+'='*60); print('5. TEXT LENGTH')
wc = txt.str.split().str.len()
print(f'chars  -> {df["Text"].astype(str).str.len().describe()[["mean","50%","max"]].round(1).to_dict()}')
print(f'words  -> {wc.describe()[["mean","50%","max"]].round(1).to_dict()}')
print(f'<=2 words: {(wc <= 2).sum()}   <=1 word: {(wc <= 1).sum()}')

# ---------- 6. Class balance ----------
print('\n'+'='*60); print('6. CLASS BALANCE (Objective 1 target)')
vc = df['Label'].value_counts()
print(pd.DataFrame({'count': vc, 'pct': (vc/len(df)*100).round(2)}).to_string())
print(f'imbalance ratio (max/min): {vc.max()/vc.min():.1f} : 1')

# ---------- 7. Reaction count ----------
print('\n'+'='*60); print('7. REACTION COUNT (Objective 2 target)')
y = df['Comment React Number']
print(f'zeros : {(y==0).sum()} ({(y==0).mean()*100:.1f}%)   <=1 : {(y<=1).mean()*100:.1f}%   <=3 : {(y<=3).mean()*100:.1f}%')
print(f'unique values: {y.nunique()}   max: {y.max():.0f}')
print(y.quantile([.5,.75,.9,.95,.99,1.0]).to_string())

# ---------- 8. Metadata (EDA only — NOT model features) ----------
print('\n'+'='*60); print('8. METADATA (EDA only)')
print(f"Category: {df['Category'].value_counts().to_dict()}")
print(f"Gender  : {df['Gender'].value_counts().to_dict()}   <- check casing inconsistency")

# ---------- 9. Noise indicators ----------
print('\n'+'='*60); print('9. NOISE INDICATORS')
print(f"leading/trailing whitespace : {(df['Text'].astype(str) != txt).sum()}")
print(f"contains URL                : {txt.str.contains(r'http|www\\.').sum()}")
print(f"contains newline            : {txt.str.contains('\\n').sum()}")
print(f"rows with <50% Bangla chars : "
      f"{txt.map(lambda s: len(re.findall(r'[\u0980-\u09FF]', s)) / max(len(re.sub(r'\\s','',s)), 1) < 0.5).sum()}")

print('\n'+'='*60)
print('Validation complete. df loaded UNMODIFIED — cleaning in next cell.')
